# Hyperparameter $D$: paired validation study

This notebook selects spline resolution without using the final test set. Within every repetition, the same fitted models are evaluated by (i) held-out particles from the training particle systems and (ii) newly simulated independent particle systems. The paired design isolates the effect of particle dependence.

## 1. Protocol and success criterion

Candidate values are compared using validation classification accuracy and empirical drift MSE. We report per-repetition winners, selection frequencies, mean performance, and the one-standard-error recommendation. A sensitivity study supports a parsimonious default; it does not prove universal optimality.

In [ ]:
from __future__ import annotations
from pathlib import Path
import csv, sys
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from mkv_classification.config import d_selection_profile
from mkv_classification.experiments import run_d_selection

PROFILE = 'smoke'  # Change only this line to 'paper' for the full sensitivity study.
config = d_selection_profile(PROFILE)
output_dir = ROOT / 'results' / f'D_selection_{PROFILE}'
config

## 2. Run or resume paired validation

In [ ]:
rows = run_d_selection(config, output_dir)
print(f'{len(rows)} D-validation rows available in {output_dir}.')

## 3. One-standard-error recommendations

In [ ]:
with (output_dir / 'D_one_se_recommendations.csv').open(newline='', encoding='utf-8') as handle:
    recommendations = list(csv.DictReader(handle))
for row in recommendations:
    print(row)

In [ ]:
with (output_dir / 'D_selection_frequencies.csv').open(newline='', encoding='utf-8') as handle:
    frequencies = list(csv.DictReader(handle))
classification = [row for row in frequencies if row['metric'] == 'classification']
strategies = ['within', 'independent']
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.2), sharey=True)
for axis, strategy in zip(axes, strategies):
    subset = [row for row in classification if row['strategy'] == strategy]
    for D in config.D_candidates:
        values = [float(row['frequency']) for row in subset if int(row['D']) == D]
        if values: axis.scatter([D] * len(values), values, label=f'D={D}')
    axis.set_title(strategy); axis.set_xlabel('D'); axis.set_ylim(0, 1); axis.grid(alpha=.2)
axes[0].set_ylabel('Selection frequency')
fig.tight_layout(); figure_path = output_dir / 'D_selection_frequency.pdf'
fig.savefig(figure_path, bbox_inches='tight'); plt.show(); print(figure_path)

## 4. Interpretation

Fixing $D=2$ in the main experiment is justified when it has the strongest aggregate validation performance and is selected by the one-standard-error rule across the relevant configurations. The manuscript should not claim that $D=2$ wins every repetition. If within-system and independent-system conclusions disagree, the independent-system result governs.